# Week 32, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w032d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from dataclasses import dataclass, field

@dataclass
class Team:
    name: str
    players: list = field(default_factory=list)
    rank: int = field(default=0, compare=False)

a, b = Team("Reds"), Team("Reds", rank=3)
a.players.append("Ada")
print(a == b, b, Team("X").players)

**Predict before you run:** what will this print?



In [ ]:
from enum import Enum

class Size(Enum):
    S = "small"
    M = "medium"
    L = "large"

print(Size("medium"), Size["L"].value, Size.S == "small", [s.name for s in Size])

**Predict before you run:** what will this print?



In [ ]:
from typing import NamedTuple

class Pair(NamedTuple):
    left: int
    right: int = 0

p = Pair(4)
print(p, p == (4, 0), p + (1,), p._replace(right=9))

**Predict before you run:** what will this print?



In [ ]:
from abc import ABC, abstractmethod

class Base(ABC):
    @abstractmethod
    def go(self): ...

class Half(Base):
    pass

class Full(Base):
    def go(self):
        return "going"

for cls in (Half, Full):
    try:
        print(cls().go())
    except TypeError:
        print("can't make", cls.__name__)

**Predict before you run:** what will this print?



In [ ]:
from typing import Protocol, runtime_checkable

@runtime_checkable
class Opens(Protocol):
    def open(self): ...

class Door:
    def open(self):
        return "creak"

print(isinstance(Door(), Opens), isinstance("door", Opens), issubclass(Door, object))

### Exercise: Frozen settings with a theme

Write an Enum `Theme` (`LIGHT = "light"`, `DARK = "dark"`, `HIGH_CONTRAST = "high contrast"`) and a frozen dataclass `Settings` with `theme` (default `Theme.LIGHT`) and `font_size` (default 16, a whole number from 8 to 32, otherwise `ValueError`). Add `with_theme(name)`, which returns a **new** `Settings` using the theme with that name in any case (`"dark"`, `"High_Contrast"`), raising `ValueError` (not `KeyError`) for an unknown name, and `bigger(step=2)`, which returns new settings with the font size increased but never above 32.

In [ ]:
from dataclasses import dataclass, replace
from enum import Enum

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_settings():\n    \"settings\"\n    S, T = ns[\"Settings\"], ns[\"Theme\"]\n    s = S()\n    assert s.theme is T.LIGHT and s.font_size == 16 and s.with_theme(\"dark\").theme is T.DARK and s.theme is T.LIGHT\n    assert s.with_theme(\"High_Contrast\").theme is T.HIGH_CONTRAST and s.bigger().font_size == 18 and S(font_size=31).bigger().font_size == 32\n    assert len({S(), S()}) == 1\n    for f in [lambda: S(font_size=7), lambda: S(font_size=33), lambda: s.with_theme(\"sepia\")]:\n        try:\n            f()\n            assert False, \"should raise ValueError\"\n        except ValueError:\n            pass\n    try:\n        s.font_size = 20\n        assert False, \"settings are frozen\"\n    except AttributeError:\n        pass")

### Exercise: Places on a map

Write a NamedTuple `LatLon` with fields `lat` and `lon`. Add a class method `parse(text)` reading `"6.45, 3.39"` (spaces allowed), raising `ValueError` unless there are two numbers with `lat` from -90 to 90 and `lon` from -180 to 180, and a method `distance_km(other)`: the great-circle distance, rounded to 1 place, using the haversine formula with the Earth's radius as 6371 km:

```text
a = sin²(Δlat / 2) + cos(lat1) × cos(lat2) × sin²(Δlon / 2)
distance = 2 × 6371 × asin(√a)
```

(with all angles in radians: `math.radians`).

In [ ]:
import math
from typing import NamedTuple

class LatLon(NamedTuple):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_places():\n    \"places\"\n    L = ns[\"LatLon\"]\n    lagos, abuja = L.parse(\"6.45, 3.39\"), L.parse(\"9.06,7.49\")\n    assert lagos == (6.45, 3.39) and lagos.distance_km(abuja) == 536.9 and abuja.distance_km(lagos) == 536.9\n    assert L(51.51, -0.13).distance_km(L(53.80, -1.55)) == 272.0 and lagos.distance_km(lagos) == 0.0\n    for bad in [\"91, 0\", \"0, 181\", \"6.45\", \"a, b\", \"1, 2, 3\"]:\n        try:\n            L.parse(bad)\n            assert False, f\"{bad!r} should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: Forge: a parcel tracker

Write `Status`, `Service`, `Scan`, `Parcel`, `Pricer`, `FlatPricer`, `WeightPricer`, `Notifier` and `Tracker` to the specification above.

In [ ]:
import math
import re
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import NamedTuple, Protocol, runtime_checkable

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_status():\n    \"statuses\"\n    St = ns[\"Status\"]\n    assert [s.value for s in St] == [\"created\", \"in transit\", \"out for delivery\", \"delivered\", \"returned\"]\n    assert St.CREATED.can_move_to(St.IN_TRANSIT) and not St.CREATED.can_move_to(St.DELIVERED)\n    assert St.IN_TRANSIT.can_move_to(St.IN_TRANSIT) and St.OUT_FOR_DELIVERY.can_move_to(St.IN_TRANSIT) and not St.IN_TRANSIT.can_move_to(St.DELIVERED)\n    assert not any(St.DELIVERED.can_move_to(s) for s in St) and not any(St.RETURNED.can_move_to(s) for s in St)\n    assert [s.name for s in St if s.finished] == [\"DELIVERED\", \"RETURNED\"]\ndef test_parcel():\n    \"parcels and scans\"\n    P, St, Sv, Sc = ns[\"Parcel\"], ns[\"Status\"], ns[\"Service\"], ns[\"Scan\"]\n    p = P(\"AB123456\", 1500)\n    assert p.status is St.CREATED and p.service is Sv.STANDARD and p.scans == [] and P(\"CD000001\", 1).scans is not p.scans\n    s = p.add_scan(\"2026-11-03 09:00\", \"Ikeja depot\", St.IN_TRANSIT)\n    assert s == Sc(\"2026-11-03 09:00\", \"Ikeja depot\", St.IN_TRANSIT) and s.place == \"Ikeja depot\"\n    p.add_scan(\"2026-11-03 14:30\", \"Yaba\", \"out for delivery\")\n    assert p.status is St.OUT_FOR_DELIVERY and len(p.scans) == 2\n    for args in [(\"2026-11-03 14:00\", \"x\", St.DELIVERED), (\"2026-11-03 15:00\", \"x\", St.CREATED), (\"3 Nov 16:00\", \"x\", St.DELIVERED),\n                 (\"2026-11-03 16:00\", \"x\", \"lost\")]:\n        try:\n            p.add_scan(*args)\n            assert False, f\"add_scan{args} should raise ValueError\"\n        except ValueError:\n            pass\n    assert len(p.scans) == 2\n    p.add_scan(\"2026-11-03 16:05\", \"Yaba\", St.DELIVERED)\n    assert p.status.finished\n    for bad in [(\"ab123456\", 10), (\"AB12345\", 10), (\"AB1234567\", 10), (\"AB123456\", 0), (\"AB123456\", 2.5)]:\n        try:\n            P(*bad)\n            assert False, f\"Parcel{bad} should raise ValueError\"\n        except ValueError:\n            pass\ndef test_pricing():\n    \"pricing\"\n    P, Sv, Pr, F, W = ns[\"Parcel\"], ns[\"Service\"], ns[\"Pricer\"], ns[\"FlatPricer\"], ns[\"WeightPricer\"]\n    a, b = P(\"AA000001\", 500), P(\"AA000002\", 2001, Sv.EXPRESS)\n    assert F(399, 899).price(a) == 399 and F(399, 899).price(b) == 899 and F(399, 899).quote([a, b]) == 1298\n    assert W(250, 500).price(a) == 250 and W(250, 500).price(b) == 1250 and W(250, 500).quote([]) == 0\n    assert issubclass(F, Pr) and issubclass(W, Pr) and \"quote\" not in vars(F)\n    try:\n        Pr()\n        assert False, \"Pricer is abstract\"\n    except TypeError:\n        pass\ndef test_tracker():\n    \"the tracker\"\n    P, St, T, F, N = ns[\"Parcel\"], ns[\"Status\"], ns[\"Tracker\"], ns[\"FlatPricer\"], ns[\"Notifier\"]\n    class Inbox:\n        def __init__(self):\n            self.messages = []\n        def notify(self, code, message):\n            self.messages.append(message)\n    inbox, other = Inbox(), Inbox()\n    hs = [inbox, other]\n    t = T(F(400, 900), hs)\n    hs.clear()\n    t.register(P(\"AB123456\", 800))\n    t.register(P(\"CD654321\", 3000))\n    t.register(P(\"EF111111\", 100))\n    s = t.scan(\"AB123456\", \"2026-11-03 09:00\", \"Ikeja depot\", St.IN_TRANSIT)\n    t.scan(\"AB123456\", \"2026-11-03 13:00\", \"Ikeja\", \"out for delivery\")\n    t.scan(\"CD654321\", \"2026-11-03 10:00\", \"Kano depot\", \"in transit\")\n    assert s.status is St.IN_TRANSIT and inbox.messages == [\"AB123456: in transit at Ikeja depot\", \"AB123456: out for delivery at Ikeja\", \"CD654321: in transit at Kano depot\"]\n    assert other.messages == inbox.messages and isinstance(inbox, N)\n    assert t.by_status() == {St.OUT_FOR_DELIVERY: [\"AB123456\"], St.IN_TRANSIT: [\"CD654321\"], St.CREATED: [\"EF111111\"]}\n    assert t.bill() == 1200\n    try:\n        t.scan(\"ZZ999999\", \"2026-11-03 11:00\", \"x\", \"in transit\")\n        assert False, \"an unknown code should raise KeyError\"\n    except KeyError:\n        pass\n    try:\n        t.register(P(\"AB123456\", 5))\n        assert False, \"a duplicate code should raise ValueError\"\n    except ValueError:\n        pass\n    for bad in [lambda: T(\"flat\"), lambda: T(F(1, 2), [\"log\"]), lambda: T(F(1, 2), [object()])]:\n        try:\n            bad()\n            assert False, \"should raise TypeError\"\n        except TypeError:\n            pass")

> **One tool per job.** Look at which tool did which job, and why. The **Enums** made the statuses and services impossible to misspell, and put the rules for moving between statuses next to them. The **named tuple** was the right weight for a scan: a small immutable record that nobody needs to change. The **dataclass** wrote the parcel's routine methods and gave `__post_init__` one place for its checks. The **ABC** suited the pricers, a family the company designs itself, where forgetting `price` should fail at once. The **Protocol** suited the notifiers, which other teams write, and which shouldn't have to inherit anything from you. None of them is better than the others; each fits a different kind of need.

### Exercise: Forge bonus: an immutable playlist

Build an immutable playlist from this week's tools.

- `Genre(Enum)`: `AFROBEATS`, `HIGHLIFE`, `JAZZ`, `GRIME`, `FOLK`, with lower-case strings as values.
- `Track`, a frozen dataclass with `title`, `artist`, `seconds` (an `int` above 0, otherwise `ValueError`) and `genre` (a `Genre`, or its value as a string, which `__post_init__` turns into a member; an unknown genre raises `ValueError`). `str` gives `Ye by Burna Boy (3:51)`.
- `Playlist(name, tracks=())`, a `collections.abc.Sequence` of tracks, which can't be changed after it's made (store the tracks as a tuple). Write `__getitem__` (a slice gives a new `Playlist` with the same name) and `__len__`, and also:
  - `+` with another `Playlist`: a new playlist named after the left one, with the tracks of both;
  - a property `duration`: the total length as `"h:mm:ss"` (hours always shown, minutes and seconds as two digits);
  - `by_genre()`: a dictionary from each `Genre` that appears to a list of titles, in playlist order;
  - `shuffled(seed)`: a new playlist with the same tracks in a random order from `random.Random(seed).sample(...)`, so the same seed always gives the same order;
  - `repr` like `Playlist('Sunday', 3 tracks)`.

In [ ]:
import random
from collections.abc import Sequence
from dataclasses import dataclass
from enum import Enum

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def make():\n    T, P = ns[\"Track\"], ns[\"Playlist\"]\n    return P(\"Sunday\", [T(\"Ye\", \"Burna Boy\", 231, \"afrobeats\"), T(\"Sweet Mother\", \"Prince Nico Mbarga\", 822, \"highlife\"),\n                        T(\"Calm Down\", \"Rema\", 239, \"afrobeats\"), T(\"Take Five\", \"Dave Brubeck\", 324, \"jazz\")])\ndef test_track():\n    \"tracks\"\n    T, G = ns[\"Track\"], ns[\"Genre\"]\n    t = T(\"Ye\", \"Burna Boy\", 231, \"afrobeats\")\n    assert t.genre is G.AFROBEATS and str(t) == \"Ye by Burna Boy (3:51)\" and T(\"x\", \"y\", 5, G.FOLK).genre is G.FOLK\n    assert len({t, T(\"Ye\", \"Burna Boy\", 231, G.AFROBEATS)}) == 1\n    for bad in [(0, \"jazz\"), (2.5, \"jazz\"), (60, \"polka\")]:\n        try:\n            T(\"x\", \"y\", *bad)\n            assert False, f\"Track{bad} should raise ValueError\"\n        except ValueError:\n            pass\n    try:\n        t.seconds = 1\n        assert False, \"tracks are frozen\"\n    except AttributeError:\n        pass\ndef test_playlist():\n    \"the playlist\"\n    from collections.abc import Sequence as Seq\n    P, G = ns[\"Playlist\"], ns[\"Genre\"]\n    p = make()\n    assert len(p) == 4 and p[0].title == \"Ye\" and p[-1].artist == \"Dave Brubeck\" and repr(p) == \"Playlist('Sunday', 4 tracks)\"\n    assert isinstance(p, Seq) and p.index(p[2]) == 2 and p[1] in p and [t.title for t in reversed(p)][0] == \"Take Five\"\n    assert p.duration == \"0:26:56\" and P(\"e\").duration == \"0:00:00\" and (p + p).duration == \"0:53:52\"\n    part = p[1:3]\n    assert isinstance(part, P) and part.name == \"Sunday\" and [t.title for t in part] == [\"Sweet Mother\", \"Calm Down\"]\n    assert p.by_genre() == {G.AFROBEATS: [\"Ye\", \"Calm Down\"], G.HIGHLIFE: [\"Sweet Mother\"], G.JAZZ: [\"Take Five\"]}\n    both = p + P(\"Extra\", [p[0]])\n    assert both.name == \"Sunday\" and len(both) == 5 and len(p) == 4\n    s1, s2 = p.shuffled(7), p.shuffled(7)\n    assert [t.title for t in s1] == [t.title for t in s2] and sorted(t.title for t in s1) == sorted(t.title for t in p)\n    assert [t.title for t in p] == [\"Ye\", \"Sweet Mother\", \"Calm Down\", \"Take Five\"], \"shuffling mustn't change the original\"\n    try:\n        p + [p[0]]\n        assert False, \"adding a list should raise TypeError\"\n    except TypeError:\n        pass\n    assert not hasattr(p, \"append\") and not hasattr(p, \"__setitem__\"), \"playlists can't be changed\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [dataclasses](w032d1.html), [Enums](w032d2.html), [NamedTuple](w032d3.html), [abstract base classes](w032d4.html) or [Protocols](w032d5.html).

## Mixed review

## Forge: a parcel tracker

A delivery company scans each parcel as it moves: collected, at a depot, on the van, delivered. Customers want to know where their parcel is, the company wants a bill for each customer, and several systems (text messages, a website, a log) want to hear about every scan. Here's the specification. Each piece uses the tool that fits it.

**`Status`, an Enum** with members `CREATED`, `IN_TRANSIT`, `OUT_FOR_DELIVERY`, `DELIVERED` and `RETURNED`, whose values are `"created"`, `"in transit"`, `"out for delivery"`, `"delivered"` and `"returned"`.

- `can_move_to(other)` says whether a parcel may move from this status to `other`: from `CREATED` only to `IN_TRANSIT`; from `IN_TRANSIT` to `IN_TRANSIT` (another depot), `OUT_FOR_DELIVERY` or `RETURNED`; from `OUT_FOR_DELIVERY` to `DELIVERED`, `IN_TRANSIT` (back to the depot) or `RETURNED`; from `DELIVERED` and `RETURNED`, nowhere.
- A property `finished`: True for `DELIVERED` and `RETURNED`.

**`Service`, an Enum**: `STANDARD = "standard"`, `EXPRESS = "express"`.

**`Scan`, a NamedTuple** with fields `when` (text like `"2026-11-03 09:00"`), `place` and `status`.

**`Parcel`, a dataclass** with `code`, `grams`, `service` (default `Service.STANDARD`) and `scans` (a new empty list for each parcel).

- In `__post_init__`: `code` must be two capital letters and six digits (like `"AB123456"`), and `grams` an `int` above 0; otherwise `ValueError`.
- A property `status`: `Status.CREATED` if there are no scans, otherwise the last scan's status.
- `add_scan(when, place, status)`: `status` may be a `Status` or its value (`"in transit"`); `when` must have the form `YYYY-MM-DD HH:MM` (check it with `datetime.strptime(when, "%Y-%m-%d %H:%M")`) and be **later** than the last scan's; the move must be allowed. Any problem raises `ValueError` and adds nothing. Otherwise append and return the new `Scan`.

**`Pricer`, an ABC**, for a family of pricing schemes the company designs itself.

- An abstract method `price(parcel)`, returning pence.
- An ordinary method `quote(parcels)`: the total price of all of them.
- `FlatPricer(standard, express)`: a fixed price for each service.
- `WeightPricer(per_kg, express_extra)`: `per_kg` for each kilogram or part of one (`math.ceil(grams / 1000)`), plus `express_extra` for express parcels.

**`Notifier`, a runtime-checkable Protocol**, for anything that wants to hear about scans, whoever wrote it: a method `notify(code: str, message: str)`.

**`Tracker(pricer, notifiers=())`**, which holds everything together.

- `pricer` must be a `Pricer` and every notifier must match `Notifier`; otherwise `TypeError`. The tracker keeps its own list of notifiers.
- `register(parcel)` adds a parcel (a second parcel with the same code raises `ValueError`) and returns it.
- `scan(code, when, place, status)`: an unknown code raises `KeyError`. Otherwise it adds the scan to the parcel, then calls `notify(code, message)` on every notifier with a message like `AB123456: out for delivery at Ikeja`, and returns the scan.
- `by_status()`: a dictionary from each status that at least one parcel has to a sorted list of those parcels' codes.
- `bill()`: the pricer's quote for all the registered parcels.

## Forge bonus: an immutable playlist

## Weekly quiz

## Week 32 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can write dataclasses with defaults, `field()`, `__post_init__`, and the `frozen`, `order` and `slots` options. ([Day 1](w032d1.html))
- I can define Enums, look members up by value and name, give them behaviour, and choose `IntEnum`, `StrEnum` or `Flag` when they fit. ([Day 2](w032d2.html))
- I can write NamedTuples, use their tuple powers, avoid their pitfalls, and choose the right record type. ([Day 3](w032d3.html))
- I can write ABCs with abstract methods, and build containers from `collections.abc`. ([Day 4](w032d4.html))
- I can write Protocols, check them at run time knowing the limits, and choose between a Protocol and an ABC. ([Day 5](w032d5.html))
- I built a parcel tracker and an immutable playlist with all of them. (Today)

Tomorrow is a **rest day**. Next week: **type hints**, where the annotations you've been writing all week get checked by a tool before your program runs.